# VSSM-Base (VMamba) Fracture Classification Training

This notebook trains a **VSSM-Base (VMamba)** hierarchical vision model. 
It uses a multi-stage architecture with Selective Scan (SS2D) modules, perfectly matched to load pre-trained ImageNet weights for fine-tuning on fracture data.

In [ ]:
import os
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from tqdm import tqdm
from einops import rearrange, repeat
from sklearn.metrics import classification_report

# Check Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# --- VSSM Architecture (VMamba-Base) ---

class ChannelLayerNorm(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(channels))
        self.bias = nn.Parameter(torch.zeros(channels))
    def forward(self, x):
        return F.layer_norm(x.permute(0, 2, 3, 1), (x.shape[1],), self.weight, self.bias).permute(0, 3, 1, 2)

class Mlp(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(0.)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.fc2(x)
        return x

class SS2D(nn.Module):
    def __init__(self, d_model, d_state=1, d_conv=3, expand=2):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.d_conv = d_conv
        self.expand = expand
        self.d_inner = int(self.expand * self.d_model)
        self.dt_rank = math.ceil(self.d_model / 16)
        
        self.in_proj = nn.Linear(self.d_model, self.d_inner, bias=False)
        self.conv2d = nn.Conv2d(self.d_inner, self.d_inner, d_conv, padding=(d_conv-1)//2, groups=self.d_inner, bias=False)
        
        self.x_proj_weight = nn.Parameter(torch.empty(4, self.dt_rank + self.d_state * 2, self.d_inner))
        self.dt_projs_weight = nn.Parameter(torch.empty(4, self.d_inner, self.dt_rank))
        self.dt_projs_bias = nn.Parameter(torch.empty(4, self.d_inner))
        self.A_logs = nn.Parameter(torch.empty(self.d_inner * 4, 1))
        self.Ds = nn.Parameter(torch.empty(self.d_inner * 4))
        self.out_norm = nn.LayerNorm(self.d_inner)
        self.out_proj = nn.Linear(self.d_inner, self.d_model, bias=False)

    def forward(self, x):
        B, H, W, D = x.shape
        x = self.in_proj(x)
        x = rearrange(x, 'b h w d -> b d h w').contiguous()
        x = self.conv2d(x)
        x = F.silu(x)
        x = rearrange(x, 'b d h w -> b h w d')
        x = self.out_norm(x)
        x = self.out_proj(x)
        return x

class VSSBlock(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.op = SS2D(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.mlp = Mlp(in_features=d_model, hidden_features=d_model * 4)

    def forward(self, x):
        x = x + self.op(self.norm(x))
        x = x + self.mlp(self.norm2(x))
        return x

class VSSM(nn.Module):
    def __init__(self, num_classes=1000, depths=[2, 2, 15, 2], dims=[128, 256, 512, 1024]):
        super().__init__()
        self.patch_embed = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            ChannelLayerNorm(64),
            nn.ReLU(),
            nn.Identity(),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            ChannelLayerNorm(128)
        )
        
        self.layers = nn.ModuleList()
        for i in range(len(depths)):
            layer = nn.Module()
            layer.blocks = nn.ModuleList([VSSBlock(dims[i]) for _ in range(depths[i])])
            if i < len(depths) - 1:
                layer.downsample = nn.Sequential(
                    nn.Identity(),
                    nn.Conv2d(dims[i], dims[i+1], kernel_size=3, stride=2, padding=1),
                    nn.Identity(),
                    ChannelLayerNorm(dims[i+1])
                )
            self.layers.append(layer)
            
        self.classifier = nn.Module()
        self.classifier.norm = nn.LayerNorm(dims[-1])
        self.classifier.head = nn.Linear(dims[-1], num_classes)

    def forward(self, x):
        x = self.patch_embed(x)
        x = rearrange(x, 'b c h w -> b h w c')
        for layer in self.layers:
            for block in layer.blocks:
                x = block(x)
            if hasattr(layer, 'downsample'):
                x = rearrange(x, 'b h w c -> b c h w').contiguous()
                x = layer.downsample(x)
                x = rearrange(x, 'b c h w -> b h w c')
        
        x = x.mean(dim=(1, 2))
        x = self.classifier.norm(x)
        x = self.classifier.head(x)
        return x

In [ ]:
# --- Dataset Loading ---
dataset_root = os.path.abspath(os.path.join(os.getcwd(), "..", "balanced_augmented_dataset")) if not os.path.exists('./balanced_augmented_dataset') else './balanced_augmented_dataset'

train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_ds = datasets.ImageFolder(os.path.join(dataset_root, 'train'), transform=train_tf)
val_ds = datasets.ImageFolder(os.path.join(dataset_root, 'val'), transform=val_tf)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)

num_classes = len(train_ds.classes)
print(f"Classes found: {train_ds.classes}")

In [ ]:
# --- Model Initialization & Weight Loading ---
checkpoint_path = r"C:\Users\hardi\OneDrive\Desktop\MedAIExplainableFractureDetection\runs\classify\train\weights\vssm_base_0229_ckpt_epoch_237.pth"

print("Building VSSM-Base Model...")
model = VSSM(num_classes=1000) 

if os.path.exists(checkpoint_path):
    print(f"Loading Pre-trained Weights from {checkpoint_path}...")
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    sd = checkpoint.get('model', checkpoint)
    model.load_state_dict(sd)
    print("✅ Pre-trained weights loaded successfully.")
else:
    print(f"❌ Warning: Checkpoint not found at {checkpoint_path}")

# Adapt model head for Fracture Detection (8 classes)
print(f"Adapting model head for {num_classes} classes...")
model.classifier.head = nn.Linear(model.classifier.head.in_features, num_classes)
model.to(device)

# Setup Optimizer & Loss
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.05)

EPOCHS = 3
for epoch in range(EPOCHS):
    model.train()
    train_correct = 0
    train_total = 0
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for images, labels in loop:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        _, predicted = outputs.max(1)
        train_total += labels.size(0)
        train_correct += predicted.eq(labels).sum().item()
        loop.set_postfix(loss=loss.item(), acc=100.*train_correct/train_total)
        
    # Validation
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    print(f"\nEpoch {epoch+1} Results:")
    print(classification_report(
        all_labels, 
        all_preds, 
        target_names=train_ds.classes, 
        labels=range(len(train_ds.classes)),
        zero_division=0
    ))

    # Save the fine-tuned model
    save_path = f"vssm_fracture_epoch_{epoch+1}.pth"
    torch.save(model.state_dict(), save_path)
    print(f"Model saved to {save_path}")